# Combined NIRKSHON TB Screening Pipeline\n\nThis notebook executes the full 12-stage NIRKSHON pipeline in a single document:\n1. Dataset preparation\n2. Lung segmentation & preprocessing\n3. Model training (teacher EfficientNetV2-M + student NirikNet with knowledge distillation)\n4. Explainability (Grad‑CAM family + Consensus CAM)\n5. Model saving & deployment readiness\n\nAll code is derived from the original modular notebooks but combined for end‑to‑end reproducibility.

In [ ]:
import os\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport tensorflow as tf\nfrom tensorflow import keras\nfrom tensorflow.keras import layers\nimport matplotlib.pyplot as plt\nimport sys\n

In [ ]:
# Configuration constants\nIMG_CLS = 384          # Classification model input size\nIMG_SEG = 384          # Segmentation model input size\nBATCH_SIZE = 16\nAUTOTUNE = tf.data.AUTOTUNE\nSEED = 42\nEPOCHS = 75\nCLAHE_CLIP_LIMIT = 2.0\nCLAHE_TILE_GRID_SIZE = (8, 8)\nBORDER_PADDING_PCT = 0.05  # 5% border padding\nDROPOUT_RATE = 0.5\nLEARNING_RATE = 1e-3\nWEIGHT_DECAY = 1e-4\nCLIPNORM = 1.0\nTEMPERATURE = 4.0\nALPHA = 0.5  # Balance between hard and soft loss in knowledge distillation\n\n# Class names\nclass_names = ['Normal', 'Tuberculosis']\n\n# Set base directory for file paths\ntry:\n    BASE_DIR = os.path.dirname(os.path.abspath(__file__))\nexcept NameError:\n    BASE_DIR = \"/kaggle/working\"\n\nprint(f\"Base directory: {BASE_DIR}\")\n

In [ ]:
# Helper functions for image I/O and preprocessing\n# (copied from highres_gradcam_helper.py)\ndef read_any(path, is_dicom=False):\n    \"\"\"Read image from path, handling DICOM and regular images.\"\"\"\n    if not os.path.exists(path):\n        return None\n    try:\n        if is_dicom or path.lower().endswith('.dcm'):\n            import pydicom\n            ds = pydicom.dcmread(path)\n            img = ds.pixel_array.astype(np.float32)\n            # Normalize to 0-255\n            img = np.clip(img, 0, 255).astype(np.uint8)\n        else:\n            img = cv2.imread(path, cv2.IMREAD_UNCHANGED)\n            if img is None:\n                return None\n    except Exception as e:\n        print(f\"Error reading {path}: {e}\")\n        return None\n    # Ensure 2D grayscale\n    if len(img.shape) == 3:\n        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)\n    return img\n\ndef apply_lung_mask(img, mask):\n    \"\"\"Apply lung mask to image (zero‑out background).\"\"\"\n    if img.shape[:2] != mask.shape[:2]:\n        mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)\n    return cv2.bitwise_and(img, img, mask=mask)\n\ndef get_lung_mask_from_path(img_path):\n    \"\"\"Load a lung mask from file (expects binary mask).\"\"\"\n    mask = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)\n    if mask is None:\n        return None\n    # Ensure binary\n    _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)\n    return mask\n\ndef apply_clahe(img):\n    \"\"\"Apply CLAHE contrast limited adaptive histogram equalization.\"\"\"\n    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP_LIMIT, tileGridSize=CLAHE_TILE_GRID_SIZE)\n    return clahe.apply(img)\n\ndef largest_connected_component(mask):\n    \"\"\"Find the largest connected component in a binary mask.\"\"\"\n    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)\n    if num_labels <= 1:  # Only background\n        return mask\n    # Find the label with the largest area (excluding background label 0)\n    largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])\n    # Create mask for the largest connected component\n    lcc = np.where(labels == largest_label, 255, 0).astype(np.uint8)\n    return lcc\n\ndef morphological_cleanup(mask, kernel_size=5):\n    \"\"\"Apply morphological opening and closing to clean up the mask.\"\"\"\n    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))\n    opened = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)\n    closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)\n    return closed\n\ndef add_border_padding(image, mask, padding_ratio=BORDER_PADDING_PCT):\n    \"\"\"Add approximately 5% border padding around the mask and crop accordingly.\"\"\"\n    # Find bounding box of mask\n    coords = cv2.findNonZero(mask)\n    if coords is None:\n        return image, mask\n    x, y, w, h = cv2.findNonZero(mask)[:4]  # Actually need boundingRect\n    x, y, w, h = cv2.boundingRect(coords)\n    pad_x = int(w * padding_ratio)\n    pad_y = int(h * padding_ratio)\n    x1 = max(x - pad_x, 0)\n    y1 = max(y - pad_y, 0)\n    x2 = min(x + w + pad_x, image.shape[1])\n    y2 = min(y + h + pad_y, image.shape[0])\n    # Crop image and mask\n    padded_image = image[y1:y2, x1:x2]\n    padded_mask = mask[y1:y2, x1:x2]\n    return padded_image, padded_mask\n\ndef preprocess_image(img_path, lung_mask_path=None):\n    \"\"\"Apply the complete preprocessing pipeline as specified in Stage 5:\n    1. Read image\n    2. Segment lungs (use provided mask or generate via U‑Net)\n    3. Keep largest connected component\n    4. Apply morphological cleanup\n    5. Add approximately 5% border padding\n    6. Crop lungs\n    7. Apply CLAHE (Contrast Limited Adaptive Histogram Equalization)\n    8. Resize to 384 × 384 pixels\n    9. Convert grayscale image to 3‑channel RGB\n    10. Apply EfficientNetV2 preprocessing\n    \"\"\"\n    # Step 1: Read image\n    img = read_any(img_path)\n    if img is None:\n        return None\n\n    # Ensure image is 2D grayscale\n    if len(img.shape) == 3:\n        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)\n\n    # Step 2: Segment lungs\n    if lung_mask_path is not None and os.path.exists(lung_mask_path):\n        # Use provided lung mask\n        lung_mask = get_lung_mask_from_path(lung_mask_path)\n    else:\n        # For now, we'll use a simple threshold as placeholder\n        # In practice, this would use a trained U‑Net model\n        _, lung_mask = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)\n\n    # Step 3: Keep largest connected component\n    lung_mask = largest_connected_component(lung_mask)\n\n    # Step 4: Apply morphological cleanup\n    lung_mask = morphological_cleanup(lung_mask)\n\n    # Step 5 & 6: Add border padding and crop\n    img, lung_mask = add_border_padding(img, lung_mask, BORDER_PADDING_PCT)\n\n    # Apply mask to image\n    masked_img = apply_lung_mask(img, lung_mask)\n\n    # Step 7: Apply CLAHE\n    enhanced_img = apply_clahe(masked_img)\n\n    # Step 8: Resize to 384 × 384 pixels\n    resized_img = cv2.resize(enhanced_img, (IMG_CLS, IMG_CLS), interpolation=cv2.INTER_LINEAR)\n\n    # Step 9: Convert grayscale to 3‑channel RGB\n    rgb_img = cv2.cvtColor(resized_img, cv2.COLOR_GRAY2RGB)\n\n    # Step 10: Apply EfficientNetV2 preprocessing\n    # EfficientNetV2 expects inputs in [0, 255] range\n    processed_img = tf.keras.applications.efficientnet_v2.preprocess_input(\n        tf.cast(rgb_img, tf.float32)\n    )\n\n    return processed_img\n

In [ ]:
# Model building functions\n# Teacher: EfficientNetV2-M\ndef build_teacher():\n    \"\"\"Build EfficientNetV2‑M teacher model.\"\"\"\n    base = tf.keras.applications.EfficientNetV2M(\n        input_shape=(IMG_CLS, IMG_CLS, 3),\n        include_top=False,\n        weights='imagenet'\n    )\n    # Freeze backbone initially (will be unfrozen later in training script)\n    base.trainable = False\n    inputs = keras.Input(shape=(IMG_CLS, IMG_CLS, 3))\n    x = base(inputs, training=False)  # teacher in eval mode for feature extraction\n    x = layers.GlobalAveragePooling2D()(x)\n    x = layers.BatchNormalization()(x)\n    x = layers.Dense(512, activation='gelu')(x)\n    x = layers.Dropout(DROPOUT_RATE)(x)\n    outputs = layers.Dense(len(class_names), name='predictions')(x)\n    model = keras.Model(inputs, outputs, name='teacher_efficientnetv2m')\n    return model\n\n# Student: NirikNet\ndef se_block(x, r=8):\n    \"\"\"Squeeze‑and‑Excitation block.\"\"\"\n    filters = x.shape[-1]\n    se = layers.GlobalAveragePooling2D()(x)\n    se = layers.Reshape((1, 1, filters))(se)\n    se = layers.Dense(filters // r, activation='relu', use_bias=False)(se)\n    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)\n    return layers.multiply([x, se])\n\ndef cbam_block(feature_map, ratio=8):\n    \"\"\"Convolutional Block Attention Module.\"\"\"\n    # Channel attention\n    avg_pool = layers.GlobalAveragePooling2D()(feature_map)\n    max_pool = layers.GlobalMaxPooling2D()(feature_map)\n    avg_pool = layers.Dense(feature_map.shape[-1] // ratio, activation='relu')(avg_pool)\n    max_pool = layers.Dense(feature_map.shape[-1] // ratio, activation='relu')(max_pool)\n    avg_pool = layers.Dense(feature_map.shape[-1], activation='sigmoid')(avg_pool)\n    max_pool = layers.Dense(feature_map.shape[-1], activation='sigmoid')(max_pool)\n    channel_avg = layers.Add()([avg_pool, max_pool])\n    channel_avg = layers.Reshape((1, 1, feature_map.shape[-1]))(channel_avg)\n    channel_feature = layers.multiply([feature_map, channel_avg])\n    # Spatial attention\n    avg_pool = layers.Lambda(lambda x: tf.reduce_mean(x, axis=3, keepdims=True))(channel_feature)\n    max_pool = layers.Lambda(lambda x: tf.reduce_max(x, axis=3, keepdims=True))(channel_feature)\n    concat = layers.Concatenate(axis=3)([avg_pool, max_pool])\n    spatial = layers.Conv2D(1, kernel_size=7, strides=1, padding='same', activation='sigmoid', use_bias=False)(concat)\n    return layers.multiply([channel_feature, spatial])\n\ndef dilated_residual_block(x, filters, dilation_rate=2, stride=1):\n    \"\"\"Dilated residual block.\"\"\"\n    shortcut = x\n    if stride != 1 or int(x.shape[-1]) != filters:\n        shortcut = layers.Conv2D(filters, 1, strides=stride,\n                                padding='same', use_bias=False,\n                                kernel_initializer='he_normal')(shortcut)\n        shortcut = layers.BatchNormalization()(shortcut)\n    x = layers.Conv2D(filters, 3, strides=stride, padding='same',\n                      dilation_rate=dilation_rate, use_bias=False,\n                      kernel_initializer='he_normal')(x)\n    x = layers.BatchNormalization()(x)\n    x = layers.Activation('gelu')(x)\n    x = layers.Conv2D(filters, 3, strides=1, padding='same',\n                      dilation_rate=dilation_rate, use_bias=False,\n                      kernel_initializer='he_normal')(x)\n    x = layers.BatchNormalization()(x)\n    x = layers.Add()([x, shortcut])\n    x = layers.Activation('gelu')(x)\n    return x\n\ndef depthwise_separable_conv_block(x, filters, stride=1):\n    \"\"\"Depthwise separable convolution block.\"\"\"\n    x = layers.SeparableConv2D(filters, 3, strides=stride, padding='same',\n                               depthwise_initializer='he_normal',\n                               pointwise_initializer='he_normal')(x)\n    x = layers.BatchNormalization()(x)\n    x = layers.Activation('gelu')(x)\n    return x\n\ndef build_student():\n    \"\"\"Build NirikNet student model as per specification.\"\"\"\n    inputs = layers.Input((IMG_CLS, IMG_CLS, 3))\n    x = inputs  # CRITICAL FIX: Initialize x with inputs\n    # Note: No Rescaling layer here as preprocessing already normalized to [-1, 1] range\n    # via EfficientNetV2 preprocessing\n\n    # Stem: two 3x3 convs\n    x = layers.Conv2D(32, 3, strides=2, padding='same',\n                     use_bias=False, kernel_initializer='he_normal')(x)\n    x = layers.BatchNormalization()(x)\n    x = layers.Activation('gelu')(x)\n    x = layers.Conv2D(64, 3, strides=2, padding='same',\n                     use_bias=False, kernel_initializer='he_normal')(x)\n    x = layers.BatchNormalization()(x)\n    x = layers.Activation('gelu')(x)\n\n    # Residual Blocks\n    x = residual_block(x, 64, stride=1)\n    x = residual_block(x, 128, stride=2)\n    x = residual_block(x, 256, stride=2)\n\n    # SE Blocks\n    x = se_block(x, r=8)\n    # CBAM Block\n    x = cbam_block(x, ratio=8)\n\n    # Additional Residual Blocks\n    x = residual_block(x, 256, stride=1)\n    x = residual_block(x, 512, stride=2)\n\n    # Dilated Residual Blocks\n    x = dilated_residual_block(x, 512, dilation_rate=2, stride=1)\n    x = dilated_residual_block(x, 512, dilation_rate=4, stride=1)\n\n    # Depthwise Separable Convolution Blocks\n    x = depthwise_separable_conv_block(x, 512, stride=1)\n    x = depthwise_separable_conv_block(x, 512, stride=1)\n\n    # Global Average Pooling\n    x = layers.GlobalAveragePooling2D()(x)\n\n    # Fully Connected Classifier\n    x = layers.Dense(1024, activation='gelu')(x)\n    x = layers.Dropout(DROPOUT_RATE)(x)\n    x = layers.Dense(512, activation='gelu')(x)\n    x = layers.Dropout(DROPOUT_RATE)(x)\n    x = layers.Dense(256, activation='gelu')(x)\n    x = layers.Dropout(DROPOUT_RATE)(x)\n    outputs = layers.Dense(len(class_names), name='predictions')(x)\n    return keras.Model(inputs, outputs, name='niriknet')\n\n# Helper to get residual_block (needed above)\ndef residual_block(x, filters, stride=1):\n    \"\"\"Standard residual block\"\"\"\n    shortcut = x\n    if stride != 1 or int(x.shape[-1]) != filters:\n        shortcut = layers.Conv2D(filters, 1, strides=stride,\n                                padding='same', use_bias=False,\n                                kernel_initializer='he_normal')(shortcut)\n        shortcut = layers.BatchNormalization()(shortcut)\n    x = layers.Conv2D(filters, 3, strides=stride,\n                     padding='same', use_bias=False,\n                     kernel_initializer='he_normal')(x)\n    x = layers.BatchNormalization()(x)\n    x = layers.Activation('gelu')(x)\n    x = layers.Conv2D(filters, 3, strides=1, padding='same',\n                     use_bias=False,\n                     kernel_initializer='he_normal')(x)\n    x = layers.BatchNormalization()(x)\n    x = layers.Add()([x, shortcut])\n    x = layers.Activation('gelu')(x)\n    return x\n

In [ ]:
# Loss functions, training step, and metrics\nfrom tensorflow.keras import losses\n\ndef compute_loss_and_update_auc(student_logits, teacher_logits, labels, ce_loss_fn, kld_loss_fn, auc_metric):\n    \"\"\"Compute combined loss (CE + KD) and update AUC metric.\"\"\"\n    # Student loss (cross‑entropy with label smoothing)\n    ce_loss = ce_loss_fn(labels, student_logits)\n    # Distillation loss (KL divergence)\n    kld_loss = kld_loss_fn(\n        tf.nn.softmax(teacher_logits / TEMPERATURE, axis=1),\n        tf.nn.log_softmax(student_logits / TEMPERATURE, axis=1)\n    )\n    # Combine losses (alpha=0.5 for hard/soft balance)\n    loss = tf.reduce_mean(ce_loss) + ALPHA * tf.reduce_mean(kld_loss)\n    # Update AUC (using student probabilities for TB class)\n    student_probs = tf.nn.softmax(student_logits, axis=-1)\n    auc_metric.update_state(labels[:, 1], student_probs[:, 1])\n    return loss\n\ndef train_step(student, teacher, images, labels, optimizer, ce_loss_fn, kld_loss_fn, auc_metric):\n    \"\"\"One training step.\"\"\"\n    with tf.GradientTape() as tape:\n        student_logits = student(images, training=True)\n        teacher_logits = teacher(images, training=False)  # teacher in eval mode\n        loss = compute_loss_and_update_auc(\n            student_logits, teacher_logits, labels, ce_loss_fn, kld_loss_fn, auc_metric\n        )\n    gradients = tape.gradient(loss, student.trainable_variables)\n    optimizer.apply_gradients(zip(gradients, student.trainable_variables))\n    return loss\n

In [ ]:
# Explainability functions\n# Grad‑CAM\ndef get_gradcam(model, img_tensor, target_layer_name=None):\n    \"\"\"Grad-CAM implementation.\"\"\"\n    if target_layer_name is None:\n        target_layer_name = get_last_conv_layer(model)\n        if target_layer_name is None:\n            target_layer_name = model.layers[-1].name\n    try:\n        target_layer = model.get_layer(target_layer_name)\n    except ValueError:\n        target_layer = model.layers[-1]\n    grad_model = tf.keras.Model(\n        [model.inputs], [target_layer.output, model.output]\n    )\n    inputs = tf.cast(img_tensor, tf.float32)\n    target_layer_outputs, predictions = grad_model(inputs)\n    pred_index = tf.argmax(predictions[0])\n    with tf.GradientTape() as tape:\n        tape.watch(inputs)\n        _, preds = grad_model(inputs)\n        loss = preds[:, pred_index]\n    grads = tape.gradient(loss, target_layer_outputs)\n    weights = tf.reduce_mean(grads, axis=(1, 2))\n    cam = tf.reduce_sum(tf.multiply(weights, target_layer_outputs), axis=-1)\n    cam = tf.maximum(cam, 0)\n    max_val = tf.reduce_max(cam)\n    if max_val == 0:\n        return tf.zeros_like(cam)\n    cam = cam / max_val\n    return cam.numpy()\n\ndef get_last_conv_layer(model):\n    \"\"\"Get the name of the last convolutional layer.\"\"\"\n    conv_layers = []\n    for layer in model.layers:\n        if isinstance(layer, layers.Conv2D) or 'conv' in layer.name.lower():\n            conv_layers.append(layer.name)\n    return conv_layers[-1] if conv_layers else None\n\ndef get_gradcam_plus(model, img_tensor, target_layer_name=None):\n    \"\"\"Grad-CAM++ implementation.\"\"\"\n    if target_layer_name is None:\n        target_layer_name = get_last_conv_layer(model)\n        if target_layer_name is None:\n            target_layer_name = model.layers[-1].name\n    try:\n        target_layer = model.get_layer(target_layer_name)\n    except ValueError:\n        target_layer = model.layers[-1]\n    grad_model = tf.keras.Model(\n        [model.inputs], [target_layer.output, model.output]\n    )\n    inputs = tf.cast(img_tensor, tf.float32)\n    target_layer_outputs, predictions = grad_model(inputs)\n    pred_index = tf.argmax(predictions[0])\n    with tf.GradientTape() as tape:\n        tape.watch(inputs)\n        _, preds = grad_model(inputs)\n        loss = preds[:, pred_index]\n    grads = tape.gradient(loss, target_layer_outputs)\n    # Gradient squared and cubed\n    grads_sq = tf.square(grads)\n    grads_cu = tf.pow(grads, 3)\n    # Global sum of gradients\n    sum_grads = tf.reduce_sum(grads, axis=(1, 2))\n    # Avoid division by zero\n    sum_grads = tf.where(sum_grads == 0, 1e-8, sum_grads)\n    alpha_num = grads_sq\n    alpha_den = 2 * grads_sq + tf.reduce_sum(\n        target_layer_outputs * grads_cu, axis=(1, 2), keepdims=True\n    )\n    alpha_den = tf.where(alpha_den == 0, 1e-8, alpha_den)\n    alphas = alpha_num / alpha_den\n    weights = tf.reduce_sum(alphas * tf.nn.relu(grads), axis=(1, 2))\n    cam = tf.reduce_sum(tf.multiply(weights, target_layer_outputs), axis=-1)\n    cam = tf.maximum(cam, 0)\n    max_val = tf.reduce_max(cam)\n    if max_val == 0:\n        return tf.zeros_like(cam)\n    cam = cam / max_val\n    return cam.numpy()\n\ndef get_layercam(model, img_tensor, target_layer_name=None):\n    \"\"\"LayerCAM implementation (using weights from forward pass).\"\"\"\n    if target_layer_name is None:\n        target_layer_name = get_second_last_conv_layer(model)\n        if target_layer_name is None:\n            target_layer_name = get_last_conv_layer(model)\n        if target_layer_name is None:\n            target_layer_name = model.layers[-1].name\n    try:\n        target_layer = model.get_layer(target_layer_name)\n    except ValueError:\n        target_layer = model.layers[-1]\n    grad_model = tf.keras.Model(\n        [model.inputs], [target_layer.output, model.output]\n    )\n    inputs = tf.cast(img_tensor, tf.float32)\n    target_layer_outputs, predictions = grad_model(inputs)\n    pred_index = tf.argmax(predictions[0])\n    with tf.GradientTape() as tape:\n        tape.watch(inputs)\n        _, preds = grad_model(inputs)\n        loss = preds[:, pred_index]\n    grads = tape.gradient(loss, target_layer_outputs)\n    pos_grads = tf.maximum(grads, 0)\n    heatmap = tf.reduce_sum(tf.multiply(pos_grads, target_layer_outputs), axis=-1)\n    heatmap = tf.maximum(heatmap, 0)\n    max_val = tf.reduce_max(heatmap)\n    if max_val == 0:\n        return tf.zeros_like(heatmap)\n    heatmap = heatmap / max_val\n    return heatmap.numpy()\n\ndef get_second_last_conv_layer(model):\n    \"\"\"Get the name of the second last convolutional layer.\"\"\"\n    conv_layers = []\n    for layer in model.layers:\n        if isinstance(layer, layers.Conv2D) or 'conv' in layer.name.lower():\n            conv_layers.append(layer.name)\n    return conv_layers[-2] if len(conv_layers) >= 2 else get_last_conv_layer(model)\n\ndef get_eigencam(model, img_tensor, target_layer_name=None):\n    \"\"\"EigenCAM implementation (using eigenvectors of activation maps).\"\"\"\n    if target_layer_name is None:\n        target_layer_name = get_third_last_conv_layer(model)\n        if target_layer_name is None:\n            target_layer_name = get_second_last_conv_layer(model)\n        if target_layer_name is None:\n            target_layer_name = get_last_conv_layer(model)\n        if target_layer_name is None:\n            target_layer_name = model.layers[-1].name\n    try:\n        target_layer = model.get_layer(target_layer_name)\n    except ValueError:\n        target_layer = model.layers[-1]\n    grad_model = tf.keras.Model(\n        [model.inputs], [target_layer.output, model.output]\n    )\n    inputs = tf.cast(img_tensor, tf.float32)\n    target_layer_outputs, _ = grad_model(inputs)\n    _, h, w, c = target_layer_outputs.shape\n    reshaped = tf.reshape(target_layer_outputs, [-1, c])\n    cov = tf.matmul(tf.transpose(reshaped), reshaped) / tf.cast(h * w, tf.float32)\n    eigenvalues, eigenvectors = tf.linalg.eigh(cov)\n    idx = tf.argsort(eigenvalues, direction='DESCENDING')\n    eigenvectors = tf.gather(eigenvectors, idx, axis=1)\n    principal_component = eigenvectors[:, 0]\n    projection = tf.matmul(reshaped, tf.expand_dims(principal_component, -1))\n    heatmap = tf.reshape(projection, [h, w])\n    heatmap = tf.maximum(heatmap, 0)\n    max_val = tf.reduce_max(heatmap)\n    if max_val == 0:\n        return tf.zeros_like(heatmap)\n    heatmap = heatmap / max_val\n    return heatmap.numpy()\n\ndef get_third_last_conv_layer(model):\n    \"\"\"Get the name of the third last convolutional layer.\"\"\"\n    conv_layers = []\n    for layer in model.layers:\n        if isinstance(layer, layers.Conv2D) or 'conv' in layer.name.lower():\n            conv_layers.append(layer.name)\n    return conv_layers[-3] if len(conv_layers) >= 3 else get_second_last_conv_layer(model)\n\ndef get_consensus_cam(model, img_tensor, target_layer_names=None):\n    \"\"\"Consensus CAM - average of normalized individual CAM outputs.\"\"\"\n    if target_layer_names is None:\n        conv_layers = get_conv_layers(model)\n        # Use last 4 distinct convolutional layers, or duplicate last if needed\n        target_layer_names = []\n        for i in range(1, 5):\n            idx = -i\n            if abs(idx) <= len(conv_layers):\n                target_layer_names.append(conv_layers[idx])\n            else:\n                target_layer_names.append(conv_layers[-1] if conv_layers else model.layers[-1].name)\n        # Remove duplicates while preserving order\n        seen = set()\n        unique_names = []\n        for name in target_layer_names:\n            if name not in seen:\n                seen.add(name)\n                unique_names.append(name)\n        target_layer_names = unique_names[:4]  # Ensure at most 4\n        # If we have less than 4, pad with the last one\n        while len(target_layer_names) < 4:\n            target_layer_names.append(target_layer_names[-1] if target_layer_names else model.layers[-1].name)\n\n    cams = []\n    for layer_name in target_layer_names:\n        try:\n            gradcam = get_gradcam(model, img_tensor, layer_name)\n            gradcam_plus = get_gradcam_plus(model, img_tensor, layer_name)\n            layercam = get_layercam(model, img_tensor, layer_name)\n            eigencam = get_eigencam(model, img_tensor, layer_name)\n            cam_stack = np.stack([gradcam, gradcam_plus, layercam, eigencam], axis=-1)\n            cam_mean = np.mean(cam_stack, axis=-1)\n            cams.append(cam_mean)\n        except Exception as e:\n            print(f\"Warning: Could not compute CAM for layer {layer_name}: {e}\")\n            continue\n    if not cams:\n        fallback_layer = get_last_conv_layer(model)\n        if fallback_layer is None:\n            fallback_layer = model.layers[-1].name\n        return get_gradcam(model, img_tensor, fallback_layer)\n    consensus_cam = np.mean(np.stack(cams, axis=0), axis=0)\n    max_val = np.max(consensus_cam)\n    if max_val == 0:\n        return np.zeros_like(consensus_cam)\n    consensus_cam = consensus_cam / max_val\n    return consensus_cam\n\ndef get_conv_layers(model):\n    \"\"\"Get list of convolutional layer names in the model.\"\"\"\n    conv_layers = []\n    for layer in model.layers:\n        if isinstance(layer, layers.Conv2D) or 'conv' in layer.name.lower():\n            conv_layers.append(layer.name)\n    return conv_layers\n

In [ ]:
# Dataset handling functions\ndef load_metadata():\n    \"\"\"Load and prepare metadata from CSV files.\n    Tries canonical location first, then legacy location.\n    Prints error and exits if not found in either location.\"\"\"\n    import sys\n    try:\n        base_dir = os.path.dirname(os.path.abspath(__file__))  # CNN Model Training directory\n    except NameError:\n        base_dir = \"/kaggle/working\"\n    project_root = os.path.dirname(base_dir)  # Go up one level from CNN Model Training to project root\n    canonical_path = os.path.join(project_root, 'datasets', 'processed', 'master_metadata.csv')\n    legacy_path = os.path.join(base_dir, 'metadata', 'master_metadata.csv')\n    if os.path.exists(canonical_path):\n        metadata_path = canonical_path\n        print(f\"Loading metadata from canonical location: {metadata_path}\")\n    elif os.path.exists(legacy_path):\n        metadata_path = legacy_path\n        print(f\"Loading metadata from legacy location: {metadata_path}\")\n    else:\n        print(\"ERROR: Metadata file not found. Please run Notebooks 1 and 2 to generate master_metadata.csv first.\")\n        print(\"Checked locations:\")\n        print(f\"  Canonical: {canonical_path}\")\n        print(f\"  Legacy:    {legacy_path}\")\n        sys.exit(1)\n    df = pd.read_csv(metadata_path)\n    required_columns = ['filepath', 'label', 'dataset_name', 'patient_id', 'lung_mask_path', 'split']\n    for col in required_columns:\n        if col not in df.columns:\n            if col == 'lung_mask_path':\n                df[col] = None\n            elif col == 'split':\n                np.random.seed(SEED)\n                n = len(df)\n                df['split'] = np.random.choice(['train', 'val', 'test'], size=n, p=[0.7, 0.15, 0.15])\n            else:\n                df[col] = 'unknown' if col != 'filepath' else ''\n    return df\n\ndef create_tf_dataset(df, split='train'):\n    \"\"\"Create a tf.data.Dataset from a dataframe split.\"\"\"\n    split_df = df[df['split'] == split].reset_index(drop=True)\n    def gen():\n        for _, row in split_df.iterrows():\n            img = preprocess_image(row['filepath'], row.get('lung_mask_path'))\n            if img is None:\n                continue\n            label = 0 if row['label'] == 'Normal' else 1\n            label_vec = tf.keras.utils.to_categorical(label, num_classes=len(class_names))\n            yield img, label_vec\n    output_signature = (\n        tf.TensorSpec(shape=(IMG_CLS, IMG_CLS, 3), dtype=tf.float32),\n        tf.TensorSpec(shape=(len(class_names),), dtype=tf.float32)\n    )\n    ds = tf.data.Dataset.from_generator(gen, output_signature=output_signature)\n    ds = ds.shuffle(buffer_size=len(split_df), seed=SEED) if split == 'train' else ds\n    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)\n    return ds\n

In [ ]:
# ====================== MAIN PIPELINE ======================\nprint(\"\\n=== Starting NIRKSHON TB Screening Pipeline ===\")\n\n# 1. Load metadata\ntry:\n    df = load_metadata()\nexcept SystemExit:\n    # load_metadata prints error and exits; we catch to stop notebook execution gracefully\n    raise\n\nprint(f\"Loaded metadata: {len(df)} total samples\")\nprint(f\"Class distribution:\\n{df['label'].value_counts()}\")\n\n# 2. Create datasets\ntrain_ds = create_tf_dataset(df, split='train')\nval_ds   = create_tf_dataset(df, split='val')\ntest_ds  = create_tf_dataset(df, split='test')\nprint(f\"Train batches: {tf.data.experimental.cardinality(train_ds)}\")\nprint(f\"Val   batches: {tf.data.experimental.cardinality(val_ds)}\")\nprint(f\"Test  batches: {tf.data.experimental.cardinality(test_ds)}\")\n\n# 3. Teacher model\nteacher_path = os.path.join(BASE_DIR, \"teacher_efficientnetv2m.keras\")\nif os.path.exists(teacher_path):\n    print(f\"Loading existing teacher model from {teacher_path}\")\n    teacher = tf.keras.models.load_model(teacher_path)\nelse:\n    print(\"Training teacher model (EfficientNetV2‑M)...\")\n    teacher = build_teacher()\n    teacher.summary()\n    teacher.compile(\n        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),\n        loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True, label_smoothing=0.05),\n        metrics=[tf.keras.metrics.AUC(name='auc'), tf.keras.metrics.CategoricalAccuracy(name='accuracy')]\n    )\n    history_teacher = teacher.fit(\n        train_ds,\n        validation_data=val_ds,\n        epochs=EPOCHS,\n        verbose=1\n    )\n    teacher.save(teacher_path)\n    print(f\"Teacher model saved to {teacher_path}\")\n\n# Freeze teacher for knowledge distillation\nteacher.trainable = False\nprint(\"Teacher model frozen for knowledge distillation\")\n\n# 4. Student model\nprint(\"\\nBuilding student model (NirikNet)...\")\n    student = build_student()\n    student.summary()\n    print(f\"Student trainable parameters: {student.count_params():,}\")\n\n# 5. Loss, optimizer, metrics for KD\nce = tf.keras.losses.CategoricalCrossentropy(from_logits=True, label_smoothing=0.05, reduction=\"none\")\nkld = tf.keras.losses.KLDivergence(reduction=\"none\")\nbase_lr = LEARNING_RATE\n# Compute total steps for warmup cosine decay\ntry:\n    train_count = tf.data.experimental.cardinality(train_ds).numpy()\n    if train_count == tf.data.experimental.cardinality(train_ds).inf:\n        # fallback: estimate from dataframe\n        train_count = len(df[df['split'] == 'train'])\nexcept:\n    train_count = len(df[df['split'] == 'train'])\ntotal_steps = tf.math.ceil(tf.cast(train_count, tf.float32) / BATCH_SIZE) * EPOCHS\nwarmup_ratio = 0.1\nwarmup_steps = tf.cast(int(total_steps * warmup_ratio), tf.float32)\n\nclass WarmUpCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):\n    def __init__(self, base_lr, total_steps, warmup_ratio=0.1):\n        super().__init__()\n        self.base_lr = base_lr\n        self.total_steps = tf.cast(total_steps, tf.float32)\n        self.warmup_steps = tf.cast(int(total_steps * warmup_ratio), tf.float32)\n    def __call__(self, step):\n        step = tf.cast(step, tf.float32)\n        warmup_lr = self.base_lr * (step / self.warmup_steps)\n        progress = (step - self.warmup_steps) / tf.maximum(1.0, self.total_steps - self.warmup_steps)\n        cosine_lr = 0.5 * self.base_lr * (1.0 + tf.cos(tf.constant(np.pi) * progress))\n        return tf.cond(step < self.warmup_steps, lambda: warmup_lr, lambda: cosine_lr)\n\nlr_sched = WarmUpCosineDecay(base_lr=base_lr, total_steps=total_steps, warmup_ratio=warmup_ratio)\nopt = tf.keras.optimizers.AdamW(learning_rate=lr_sched, weight_decay=WEIGHT_DECAY, clipnorm=CLIPNORM)\n\n# Metrics\ntrain_loss_metric = tf.keras.metrics.Mean(name='train_loss')\nval_loss_metric   = tf.keras.metrics.Mean(name='val_loss')\nval_auc_metric    = tf.keras.metrics.AUC(name='val_auc')\nval_acc_metric    = tf.keras.metrics.CategoricalAccuracy(name='val_accuracy')\n\n# 6. Training loop\nbest_val_auc = 0.0\nhistory = {'loss': [], 'val_loss': [], 'val_auc': [], 'val_accuracy': []}\nprint(f\"\\nStarting training for {EPOCHS} epochs...\")\nfor epoch in range(EPOCHS):\n    print(f\"\\nEpoch {epoch+1}/{EPOCHS}\")\n    # Training\n    train_loss_metric.reset_state()\n    for step, (images, labels) in enumerate(train_ds):\n        loss = train_step(student, teacher, images, labels, opt, ce, kld, val_auc_metric)\n        train_loss_metric.update_state(loss)\n        if step % 50 == 0:\n            print(f\"  Step {step}: loss = {train_loss_metric.result():.4f}\")\n    # Validation\n    val_loss_metric.reset_state()\n    val_auc_metric.reset_state()\n    val_acc_metric.reset_state()\n    for images, labels in val_ds:\n        student_logits = student(images, training=False)\n        teacher_logits = teacher(images, training=False)\n        loss = compute_loss_and_update_auc(\n            student_logits, teacher_logits, labels, ce, kld, val_auc_metric\n        )\n        val_loss_metric.update_state(loss)\n        preds = tf.argmax(student_logits, axis=-1)\n        labels_idx = tf.argmax(labels, axis=-1)\n        val_acc_metric.update_state(labels_idx, preds)\n    train_loss = train_loss_metric.result()\n    val_loss   = val_loss_metric.result()\n    val_auc    = val_auc_metric.result()\n    val_acc    = val_acc_metric.result()\n    history['loss'].append(float(train_loss))\n    history['val_loss'].append(float(val_loss))\n    history['val_auc'].append(float(val_auc))\n    history['val_accuracy'].append(float(val_acc))\n    print(f\"  Train Loss: {train_loss:.4f}\")\n    print(f\"  Val   Loss: {val_loss:.4f}\")\n    print(f\"  Val   AUC:  {val_auc:.4f}\")\n    print(f\"  Val   Acc:  {val_acc:.4f}\")\n    # Save best model\n    if val_auc > best_val_auc:\n        best_val_auc = val_auc\n        student.save(os.path.join(BASE_DIR, \"niriknet_best.keras\"))\n        print(f\"  -> Saved best student model (AUC: {val_auc:.4f})\")\n\n# 7. Final evaluation on test set\nprint(\"\\n=== Final Evaluation on Test Set ===\")\ntest_loss_metric = tf.keras.metrics.Mean(name='test_loss')\ntest_auc_metric  = tf.keras.metrics.AUC(name='test_auc')\ntest_acc_metric  = tf.keras.metrics.CategoricalAccuracy(name='test_accuracy')\nfor images, labels in test_ds:\n    student_logits = student(images, training=False)\n    teacher_logits = teacher(images, training=False)\n    loss = compute_loss_and_update_auc(\n        student_logits, teacher_logits, labels, ce, kld, test_auc_metric\n    )\n    test_loss_metric.update_state(loss)\n    preds = tf.argmax(student_logits, axis=-1)\n    labels_idx = tf.argmax(labels, axis=-1)\n    test_acc_metric.update_state(labels_idx, preds)\ntest_loss = test_loss_metric.result()\ntest_auc  = test_auc_metric.result()\ntest_acc  = test_acc_metric.result()\nprint(f\"Test Loss: {test_loss:.4f}\")\nprint(f\"Test AUC:  {test_auc:.4f}\")\nprint(f\"Test Acc:  {test_acc:.4f}\")\n\n# Save final model\nstudent.save(os.path.join(BASE_DIR, \"niriknet.keras\"))\nprint(f\"\\nFinal student model saved to {os.path.join(BASE_DIR, 'niriknet.keras')}\")\n\n# 8. Explainability: generate CAM visualisations for a few test samples\nprint(\"\\n=== Generating Explainability Visualisations ===\")\nexpl_dir = os.path.join(BASE_DIR, 'explainability_visualisations')\nos.makedirs(expl_dir, exist_ok=True)\n\n# Take first batch from test set\nfor images, labels in test_ds.take(1):\n    break\nimages_np = images.numpy()\nlabels_np = labels.numpy()\nnum_to_show = min(5, images_np.shape[0])\nprint(f\"Generating CAMs for {num_to_show} sample(s)...\")\nfor i in range(num_to_show):\n    img_tensor = tf.expand_dims(images_np[i], axis=0)  # shape (1, H, W, 3)\n    true_label = class_names[np.argmax(labels_np[i])]\n    pred_probs = tf.nn.softmax(student(img_tensor, training=False)).numpy()[0]\n    pred_label = class_names[np.argmax(pred_probs)]\n    pred_conf  = np.max(pred_probs)\n    \n    # Compute CAMs\n    gradcam        = get_gradcam(student, img_tensor)\n    gradcam_plus   = get_gradcam_plus(student, img_tensor)\n    layercam       = get_layercam(student, img_tensor)\n    eigencam       = get_eigencam(student, img_tensor)\n    consensus_cam  = get_consensus_cam(student, img_tensor)\n    \n    # Plot\n    fig, axes = plt.subplots(2, 3, figsize=(15, 10))\n    axes = axes.flatten()\n    # Original image (need to undo EfficientNetV2 preprocessing for display)\n    # EfficientNetV2 preprocessing: x = (x / 127.5) - 1  => to revert: x = (x + 1) * 127.5\n    img_disp = ((images_np[i] + 1.0) * 127.5).astype(np.uint8)\n    if img_disp.shape[-1] == 1:\n        img_disp = np.repeat(img_disp, 3, axis=-1)\n    axes[0].imshow(img_disp, cmap='gray')\n    axes[0].set_title(f\"Original\\nTrue: {true_label}\\\\nPred: {pred_label} ({pred_conf:.2f})\")\n    axes[0].axis('off')\n    # Grad-CAM\n    axes[1].imshow(img_disp, cmap='gray')\n    axes[1].imshow(gradcam, cmap='jet', alpha=0.5)\n    axes[1].set_title(\"Grad-CAM\")\n    axes[1].axis('off')\n    # Grad-CAM++\n    axes[2].imshow(img_disp, cmap='gray')\n    axes[2].imshow(gradcam_plus, cmap='jet', alpha=0.5)\n    axes[2].set_title(\"Grad-CAM++\")\n    axes[2].axis('off')\n    # LayerCAM\n    axes[3].imshow(img_disp, cmap='gray')\n    axes[3].imshow(layercam, cmap='jet', alpha=0.5)\n    axes[3].set_title(\"LayerCAM\")\n    axes[3].axis('off')\n    # EigenCAM\n    axes[4].imshow(img_disp, cmap='gray')\n    axes[4].imshow(eigencam, cmap='jet', alpha=0.5)\n    axes[4].set_title(\"EigenCAM\")\n    axes[4].axis('off')\n    # Consensus CAM\n    axes[5].imshow(img_disp, cmap='gray')\n    axes[5].imshow(consensus_cam, cmap='jet', alpha=0.5)\n    axes[5].set_title(\"Consensus CAM\")\n    axes[5].axis('off')\n    \n    plt.tight_layout()\n    out_path = os.path.join(expl_dir, f\"sample_{i}_true_{true_label}_pred_{pred_label}.png\")\n    plt.savefig(out_path, dpi=150)\n    plt.close()\n    print(f\"  Saved: {out_path}\")\n\nprint(f\"\\nExplainability images saved to: {expl_dir}\")\nprint(\"\\nPipeline completed successfully!\")\nprint(\"Saved models:\")\nprint(f\"  - Teacher:   {teacher_path}\")\nprint(f\"  - Best student: {os.path.join(BASE_DIR, 'niriknet_best.keras')}\")\nprint(f\"  - Final student: {os.path.join(BASE_DIR, 'niriknet.keras')}\")\n